# Module 3 - Simulating Polarisation Dynamics

**Time:** 13:45-15:15  
**Tool focus:** NDlib plus explicit bounded-confidence simulation code

The notebook keeps the simulation logic visible so participants can see exactly how confidence thresholds and biased exposure alter the opinion trajectories.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / ".cache"
PLOTS_DIR = CACHE_DIR / "plots"
(CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import matplotlib.pyplot as plt
import ndlib
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import ndlib.models.ModelConfig as mc
from ndlib.models.opinions import AlgorithmicBiasModel, HKModel
from ndlib.viz.mpl.OpinionEvolution import OpinionEvolution
from IPython.display import Image, display

sns.set_theme(context="talk", style="whitegrid")
print("NDlib version:", getattr(ndlib, "__version__", "installed"))


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


def write_demo_graph_files(seed=42):
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    graph = build_demo_graph(seed=seed)

    node_frame = pd.DataFrame(
        [{"node_id": node, **attrs} for node, attrs in graph.nodes(data=True)]
    )
    edge_frame = pd.DataFrame(
        [{"source": source, "target": target} for source, target in graph.edges()]
    )

    nx.write_graphml(graph, DATA_RAW / "workshop_network.graphml")
    nx.write_gexf(graph, DATA_RAW / "workshop_network.gexf")
    nx.write_edgelist(graph, DATA_RAW / "workshop_network.edgelist", data=False)
    node_frame.to_csv(DATA_RAW / "workshop_nodes.csv", index=False)
    edge_frame.to_csv(DATA_PROCESSED / "workshop_edges.csv", index=False)


def load_graph(path):
    path = Path(path)
    if path.suffix == ".graphml":
        graph = nx.read_graphml(path)
    elif path.suffix == ".gexf":
        graph = nx.read_gexf(path)
    else:
        graph = nx.read_edgelist(path, nodetype=int)
        node_frame = pd.read_csv(DATA_RAW / "workshop_nodes.csv")
        attributes = node_frame.set_index("node_id").to_dict(orient="index")
        nx.set_node_attributes(graph, attributes)

    graph = nx.convert_node_labels_to_integers(graph, label_attribute="original_id")
    for _, attrs in graph.nodes(data=True):
        attrs["opinion"] = float(attrs["opinion"])
        attrs["activity"] = int(attrs["activity"])
        attrs["enclave"] = int(attrs["enclave"])
    return graph


In [ ]:
write_demo_graph_files()
G = load_graph(DATA_RAW / "workshop_network.graphml")


In [ ]:
def count_clusters(opinions, tolerance=0.08):
    values = sorted(opinions.values())
    if not values:
        return 0
    clusters = 1
    for previous, current in zip(values, values[1:]):
        if abs(current - previous) > tolerance:
            clusters += 1
    return clusters


def summarise_state(label, opinions):
    values = np.array(list(opinions.values()))
    if len(values) == 0:
        return {
            "scenario": label,
            "mean": np.nan,
            "std": np.nan,
            "clusters": 0,
            "range": np.nan,
        }
    return {
        "scenario": label,
        "mean": round(float(values.mean()), 4),
        "std": round(float(values.std()), 4),
        "clusters": count_clusters(opinions),
        "range": round(float(values.max() - values.min()), 4),
    }


In [ ]:
def run_opinion_model(model_class, graph, iterations, seed=None, **parameters):
    if model_class is AlgorithmicBiasModel:
        model = model_class(graph, seed=seed)
    else:
        if seed is not None:
            np.random.seed(seed)
        model = model_class(graph)

    configuration = mc.Configuration()
    for parameter_name, parameter_value in parameters.items():
        configuration.add_model_parameter(parameter_name, parameter_value)

    model.set_initial_status(configuration)
    trends = model.iteration_bunch(iterations)
    final_status = getattr(model, "status", {})
    final_opinions = {node: float(value) for node, value in final_status.items()}
    return model, trends, final_opinions


## 1. Phase transition under different confidence thresholds


In [ ]:
epsilons = [0.45, 0.25, 0.10]
hk_runs = {}
hk_summaries = []

for epsilon in epsilons:
    model, trends, final_state = run_opinion_model(HKModel, G, iterations=80, epsilon=epsilon, seed=42 + int(epsilon * 100))
    hk_runs[epsilon] = (model, trends, final_state)
    hk_summaries.append(summarise_state(f"HK epsilon={epsilon}", final_state))

pd.DataFrame(hk_summaries)


In [ ]:
for epsilon in epsilons:
    model, trends, _ = hk_runs[epsilon]
    print(f"Opinion evolution for HK epsilon={epsilon}")
    output_path = PLOTS_DIR / f"module3_hk_epsilon_{str(epsilon).replace('.', '_')}.png"
    OpinionEvolution(model, trends).plot(filename=str(output_path))
    display(Image(filename=str(output_path)))


## 2. Add algorithmic bias to the interaction rule


`AlgorithmicBiasModel` in the installed NDlib version is stable on complete interaction graphs, so the biased/unbiased comparison below uses a complete exposure space with the same number of agents. This still matches the conceptual point of the module: selective exposure changes *who* interacts, not just *what* the current network topology looks like.


In [ ]:
complete_graph = nx.complete_graph(G.number_of_nodes())
hk_model, hk_trends, hk_state = run_opinion_model(HKModel, complete_graph, iterations=80, epsilon=0.25, seed=123)
ab_model, ab_trends, ab_state = run_opinion_model(AlgorithmicBiasModel, complete_graph, iterations=80, epsilon=0.25, gamma=7.5, seed=123)

pd.DataFrame(
    [
        summarise_state("HKModel on complete graph", hk_state),
        summarise_state("AlgorithmicBiasModel gamma=7.5", ab_state),
    ]
)


In [ ]:
print("Opinion evolution for HKModel")
hk_plot_path = PLOTS_DIR / "module3_hk_model.png"
OpinionEvolution(hk_model, hk_trends).plot(filename=str(hk_plot_path))
display(Image(filename=str(hk_plot_path)))

print("Opinion evolution for AlgorithmicBiasModel")
ab_plot_path = PLOTS_DIR / "module3_algorithmic_bias_model.png"
OpinionEvolution(ab_model, ab_trends).plot(filename=str(ab_plot_path))
display(Image(filename=str(ab_plot_path)))
